In [6]:
import spacy
import import_ipynb

In [23]:
from Assign_Labels import *

In [ ]:
nlp = spacy.load("en_core_web_lg")

In [24]:
def extract_relationships(predicted_entities, sentence):
    """
    Extract structured financial relationships using BERT-CRF labels + spaCy dependency parsing.

    Returns: List of (company, financial_property, year, monetary_value)
    """
    # doc = nlp(sentence)  # Process sentence with spaCy

    relationships = []
    last_company = None
    last_year = None
    last_property = None

    entity_dict = {
        "COMPANY": [],
        "PROPERTY": [],
        "YEAR": [],
        "VALUE": [],
        "MULTIPLIER": [],
        "UNIT": [],
    }

    for token, label in predicted_entities:
        if label.startswith("B-COMPANY"):
            entity_dict["COMPANY"].append(token)
        elif label.startswith("B-PROPERTY"):
            entity_dict["PROPERTY"].append(token)
        elif label.startswith("I-PROPERTY") and entity_dict["PROPERTY"]:
            entity_dict["PROPERTY"][-1] += " " + token.replace("##", "") 
        elif label.startswith("B-YEAR"):
            entity_dict["YEAR"].append(token)
        elif label.startswith("I-YEAR") and entity_dict["YEAR"]:
            entity_dict["YEAR"][-1] += token.replace("##", "")
        elif label.startswith("B-VALUE"):
            entity_dict["VALUE"].append(token)
        elif label.startswith("I-VALUE") and entity_dict["VALUE"]:
            entity_dict["VALUE"][-1] += " " + token  # Merge multi-token values
        elif label.startswith("B-MULTIPLIER"):
            entity_dict["MULTIPLIER"].append(token)
        elif label.startswith("B-UNIT"):
            entity_dict["UNIT"].append(token)

    # # 🔹 Step 2: Use spaCy dependency parsing to match numbers with properties & years
    # for token in doc:
    #     if token.dep_ in {"amod", "compound"} and token.head.text in entity_dict["VALUE"]:
    #         # Match "200" in "200 billion" or "$200B"
    #         index = entity_dict["VALUE"].index(token.head.text)
    #         entity_dict["VALUE"][index] = token.text + " " + entity_dict["VALUE"][index]  # Merge with modifier

    #     elif token.dep_ == "nsubj" and token.head.text in entity_dict["PROPERTY"]:
    #         # Match "revenue" to its company (e.g., "Apple’s revenue")
    #         index = entity_dict["PROPERTY"].index(token.head.text)
    #         entity_dict["PROPERTY"][index] = token.text + " " + entity_dict["PROPERTY"][index]

    # 🔹 Step 3: Match extracted values to properties & companies
    for i in range(max(len(entity_dict["COMPANY"]), len(entity_dict["PROPERTY"]), len(entity_dict["VALUE"]))):
        company = entity_dict["COMPANY"][i] if i < len(entity_dict["COMPANY"]) else last_company
        property_ = entity_dict["PROPERTY"][i] if i < len(entity_dict["PROPERTY"]) else last_property
        year = entity_dict["YEAR"][i] if i < len(entity_dict["YEAR"]) else last_year
        value = entity_dict["VALUE"][i] if i < len(entity_dict["VALUE"]) else None

        if year:
            last_year = year  # Update last known year
        if property_:
            last_property = property_
        if company:
            last_company = company

        if company and property_ and value:
            relationships.append((company, property_, year, value))

    return relationships

In [25]:
def process_sentence(sentence):
    """Extract structured financial data using BERT-CRF and dependency parsing."""
    predicted_entities = predict(sentence)  # Step 1: Run BERT-CRF for labeling
    print(predicted_entities)
    relationships = extract_relationships(predicted_entities, sentence)  # Step 2: Use spaCy dependency parsing

    company_data = {}

    for company, financial_property, year, monetary_value in relationships:
        year = year if year else "Unknown Year"

        if company not in company_data:
            company_data[company] = {}

        if year not in company_data[company]:
            company_data[company][year] = {}

    # Debugging Output
    print(f"\n🔹 Original Sentence: {sentence}")
    print(f"🔸 Extracted Relationships: {relationships}")
    print(f"🔹 Structured Output: {company_data}")

    return company_data


In [26]:
test_sentences = "In 2023, Apple’s revenue grew to 200 billion dollars, while Microsoft reported revenue of 180 billion dollars. Google’s net profit in 2022 was 50 billion dollars. Amazon’s operating expenses in 2021 totaled 150 billion dollars."
print(process_sentence(sentence))

[('[CLS]', 'O'), ('in', 'O'), ('202', 'B-YEAR'), ('##3', 'I-YEAR'), (',', 'O'), ('apple', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('revenue', 'I-PROPERTY'), ('grew', 'I-PROPERTY'), ('to', 'O'), ('200', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), (',', 'I-VALUE'), ('while', 'O'), ('microsoft', 'B-COMPANY'), ('reported', 'O'), ('revenue', 'I-PROPERTY'), ('of', 'O'), ('180', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'I-VALUE'), ('google', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('net', 'B-PROPERTY'), ('profit', 'I-PROPERTY'), ('in', 'O'), ('202', 'B-YEAR'), ('##2', 'I-YEAR'), ('was', 'O'), ('50', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'I-VALUE'), ('amazon', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('operating', 'B-PROPERTY'), ('expenses', 'I-PROPERTY'), ('in', 'O'), ('2021', 'B-YEAR'), ('totaled', 'O'), ('150', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'I-VALUE'), ('[SEP]', 'O')]

🔹 Origina